# Data Modeling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/data-modeling.ipynb)

Data modeling is where most analytics problems are actually decided: a warehouse can be fast and still report the wrong revenue if the grain, keys, or history handling are wrong. This notebook covers the Kimball vocabulary interviewers expect (grain, facts, dimensions, SCD, bridges, conformed dimensions), runnable DuckDB examples of each pattern, and the trade-offs between star, snowflake, and Data Vault.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is data modeling in analytics and why does it matter?

Data modeling structures data so business questions can be answered reliably and cheaply. An OLTP schema is modeled for correct *writes*: 3NF stores each fact once so an update cannot create a contradiction. An analytics model is modeled for correct *reads*: a fact table at a declared grain plus dimensions that describe it. The trade is write complexity for read simplicity.

```text
sources (OLTP, APIs, events, files)
  -> staging / bronze      raw, append-only, one table per source
  -> integration / silver  cleaned, conformed, deduplicated entities
  -> marts / gold          facts + dimensions at a defined grain, documented metrics
  -> BI, notebooks, reverse ETL, feature stores
```

Three decisions recur at every layer:

1. **Grain**: what does one row represent?
2. **Keys**: natural, surrogate, or composite, and how they join.
3. **Change handling**: overwrite (SCD1), version history (SCD2), or append-only events.

Normalization levels to know: 1NF makes columns atomic, 2NF removes partial-key dependencies, 3NF removes transitive dependencies. Dimensional modeling deliberately denormalizes dimensions so a report needs one join per dimension.

Interview signal: a candidate who starts with "what is the grain and what does the business call a customer?" shows design judgment; one who starts writing `JOIN`s shows only SQL.

Common mistake: copying source tables 1:1 into the warehouse and calling it a model. A 1:1 copy inherits the source grain, so every consumer re-derives the same metric and gets a different answer.

## What is grain and why is it the first question to answer?

Grain is the level of detail of a table, such as "one row per order line" or "one row per account per day". It is a contract with three parts: the entity, the event or period, and the uniqueness rule. Grain and primary key are the same statement written twice; if they disagree, the model is wrong.

Grain fixes how each measure behaves:

+ **Additive** across all dimensions: quantity, amount.
+ **Semi-additive**, usually not across time: account balance, inventory on hand. Use `avg`, `min`, `max`, or last value over time, never `sum`.
+ **Non-additive**: ratios and percentages. Store numerator and denominator; compute the ratio at query time.

Choose the **lowest grain that answers the questions** and that you can afford to store: you can always aggregate up, never split a row back apart. "Revenue by product by month and average order value" needs order-line grain, because AOV is `sum(line_amount) / count(distinct order_id)`.

Common grain mistakes:

+ Storing order-level values (shipping, tax) on every order-line row, then summing them. Keep them in a `fact_order` at order grain, or allocate them to lines with a documented rule, never both.
+ Mixing snapshot and event grain in one table so a `sum` silently multiplies.
+ No declared key, so nothing prevents duplicate loads.
+ Choosing the grain of today's dashboard instead of the business process.

The cell below builds a small star schema in DuckDB that the rest of the notebook reuses, and encodes the grain as a uniqueness test.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
create table dim_date as
select cast(strftime(d, '%Y%m%d') as int) as date_sk,
       cast(d as date)                    as date_value,
       strftime(d, '%Y-%m')               as year_month,
       dayofweek(d) in (0, 6)             as is_weekend
from range(date '2026-09-01', date '2026-11-01', interval 1 day) t(d);

create table dim_product (
  product_sk bigint primary key, product_id varchar, product_name varchar,
  category varchar, department varchar
);
insert into dim_product values
  (1, 'P1', 'Trail shoe', 'Footwear', 'Outdoor'),
  (2, 'P2', 'Rain jacket', 'Apparel', 'Outdoor'),
  (3, 'P3', 'Desk lamp', 'Lighting', 'Home');

create table fact_order_line (
  order_id varchar not null,          -- degenerate dimension
  line_number int not null,
  customer_id varchar not null,
  product_sk bigint not null,
  order_date_sk int not null,
  order_ts timestamp not null,
  quantity int not null,
  net_amount decimal(14, 2) not null,
  unique (order_id, line_number)      -- grain guard
);
insert into fact_order_line values
  ('O1', 1, 'C1', 1, 20260920, '2026-09-20 10:00', 1, 120.00),
  ('O1', 2, 'C1', 3, 20260920, '2026-09-20 10:00', 2,  60.00),
  ('O2', 1, 'C2', 2, 20261003, '2026-10-03 09:30', 1, 200.00),
  ('O3', 1, 'C1', 1, 20261015, '2026-10-15 18:45', 2, 240.00),
  ('O3', 2, 'C1', 2, 20261015, '2026-10-15 18:45', 1, 180.00);
""")

# grain test: the declared key must be unique (query must return zero rows)
dupes = con.execute("""
    select order_id, line_number, count(*) from fact_order_line
    group by all having count(*) > 1
""").fetchall()
assert dupes == []
print("fact_order_line rows:", con.execute("select count(*) from fact_order_line").fetchone()[0])

## What is the difference between fact and dimension tables?

Facts store measurable events or snapshots (orders, payments, balances) plus foreign keys to dimensions: tall, narrow, growing with events, usually partitioned by date. Dimensions store descriptive attributes (customer, product, date) used to filter and group: short, wide, growing with entities. Facts answer "what happened, how much"; dimensions answer "who, what, where, when".

Practical rules:

+ Fact foreign keys reference the dimension's surrogate key so every join is many-to-one and no fact row duplicates.
+ Not every number is a measure: tax rate, discount percentage, and latitude are descriptive and belong in a dimension.
+ Descriptive text in the fact (to "save a join") repeats per event, turning a rename into a mass update. The exception is a **degenerate dimension** such as `order_id`: an identifier at the fact grain with no attributes of its own, kept in the fact with no dimension table.
+ A **junk dimension** is the opposite move: many low-cardinality flags collapsed into one small table so the fact does not grow a column per flag.

## What is a star schema and what are its benefits?

A star schema is a central fact table joined to flat, denormalized dimensions on single-column surrogate keys. Every dimension join is many-to-one, so the fact row count never changes and measures stay additive. BI tools like stars because they can infer the join graph and generate `GROUP BY` without fan-out.

```text
          dim_date
             |
dim_product -- fact_order_line -- dim_customer
             |
          dim_store
```

The star contract is one row per dimension key. A dimension with one row per address (or per SCD2 version, joined without a filter) duplicates every fact row. Adding a second fact to the diagram does not make it a star: joining two facts at different grains fans out (see double counting below).

In [ ]:
star = con.execute("""
    select d.year_month,
           p.category,
           sum(f.net_amount) as net_revenue,
           sum(f.quantity)   as units
    from fact_order_line f
    join dim_date    d on d.date_sk = f.order_date_sk
    join dim_product p on p.product_sk = f.product_sk
    group by all
    order by all
""").fetchall()
for row in star:
    print(row)

# a many-to-one star join never changes the measure total
star_total = sum(row[2] for row in star)
fact_total = con.execute("select sum(net_amount) from fact_order_line").fetchone()[0]
assert star_total == fact_total == 800

## Star schema vs snowflake schema: what are the trade-offs?

A snowflake normalizes dimensions into sub-tables (`dim_product -> dim_category -> dim_department`). It reduces redundancy and makes a shared sub-attribute single-sourced, at the cost of more joins.

```sql
-- snowflaked
create table dim_product  (product_sk bigint primary key, product_name varchar, category_sk bigint);
create table dim_category (category_sk bigint primary key, category_name varchar, department_sk bigint);
-- star: the same attributes flattened
create table dim_product  (product_sk bigint primary key, product_name varchar,
                           category_name varchar, department_name varchar);
```

| Aspect | Star | Snowflake |
|---|---|---|
| Joins per report | 1 hop per dimension | multiple hops |
| Storage | more redundancy (compresses well) | less redundancy |
| Ease for BI users | high | lower |
| Dimension maintenance | rewrite a wide row | update a small table |
| Best fit | BI marts, self-service | huge or widely shared sub-dimensions |

On a columnar MPP engine the argument tilts further toward stars: repeated text compresses well, so the storage saving is small, and small dimensions are broadcast to every node so a one-hop join moves no fact rows, while a chain of joins adds stages and chances for a shuffle.

Interview answer: use a star in the consumption layer. Snowflake only when a sub-attribute is genuinely shared by many dimensions (global geography, cost-center hierarchy) or the dimension is enormous. If you keep a snowflake, publish flattened views generated from it, not hand-maintained copies that drift.

## What are the main types of fact tables?

| Type | Grain | Behavior | Answers |
|---|---|---|---|
| Transaction | one row per event | insert-only, sparse | "how many / how much?" |
| Periodic snapshot | one row per entity per period | dense, usually semi-additive | "what was the balance / how many open on the 15th?" |
| Accumulating snapshot | one row per process instance | updated in place as milestones complete; lag columns | "how long did each stage take?" |
| Factless | one row per event or relationship, no measure | counted, not summed | coverage, eligibility, attendance |

```sql
-- periodic snapshot: one row per account per day
create table fact_account_balance_daily (
  account_sk bigint, snapshot_date_sk int, closing_balance numeric(18, 2),
  primary key (account_sk, snapshot_date_sk)
);
-- accumulating snapshot: one row per order, milestones updated in place
create table fact_order_fulfillment (
  order_sk bigint primary key,
  ordered_date_sk int, packed_date_sk int, shipped_date_sk int, delivered_date_sk int,
  days_to_ship int
);
```

Transaction and snapshot facts cannot substitute for each other. Modeling a state question ("open tickets") as a transaction fact forces you to net deltas by hand; model it as a periodic snapshot and the metric is a row count. Accumulating snapshots make stage durations cheap (`avg(days_in_transit)`) at the cost of in-place updates that must be idempotent.

## What is a surrogate key and when should you use it?

A natural key identifies a thing in the source; a surrogate key identifies a row in your dimension, so the warehouse controls identity and versioning. Use one when source keys are unstable or recycled, when multiple sources have overlapping key spaces, and whenever you keep SCD2 history.

+ Type 1 keeps one surrogate key per entity forever; Type 2 issues a new surrogate key per version while the natural key repeats.
+ Facts store the surrogate key resolved at load time, which makes point-in-time joins exact.
+ A deterministic hash of the business key is parallel-safe (every worker computes the same value, no coordination). An identity/sequence is compact but needs a central generator.
+ Keep keys meaningless; never encode business meaning in digits.
+ Hash the **business key only**, including the source system when key spaces overlap. If the hash includes a tracked attribute like `segment`, a segment change mints a new entity identity and cross-version reporting breaks.

In [ ]:
keys = con.execute("""
    select source_system, customer_id,
           md5(source_system || '|' || customer_id) as customer_hk
    from (values ('crm', '1001'), ('billing', '1001'), ('crm', '1001')) t(source_system, customer_id)
""").fetchall()
for row in keys:
    print(row)

# deterministic: same input -> same key; source system prevents cross-system collisions
assert keys[0][2] == keys[2][2]
assert keys[0][2] != keys[1][2]

## What is a conformed dimension?

A conformed dimension is shared across facts or marts with the same table, the same surrogate keys, and the same attribute definitions, for example one `dim_customer` used by `fact_order` and `fact_support_ticket`. That sameness makes **drill-across** valid: aggregate each fact to a shared dimension attribute at its own grain, then join the small results. Never join fact to fact directly.

Keeping it conformed: one owner, one refresh schedule, one definition per attribute, and a **bus matrix** recording which processes use which conformed dimensions. `dim_date` is the most common conformed dimension.

```text
                 dim_date  dim_customer  dim_product  dim_store
fact_order_line     X          X             X            X
fact_payment        X          X                          X
fact_inventory      X                        X            X
```

Gotcha: two marts that each build their own `dim_customer` with different segment logic are not conformed even when column names match; so is conforming the keys but not the attributes. The drift stays silent until a cross-domain report disagrees.

## What is SCD (Slowly Changing Dimension)?

SCD describes how a dimension handles attribute changes over time. Pick the type **per attribute**, not per table: a name typo is Type 1, a segment change that should keep historical revenue attributed to the old segment is Type 2.

| Type | Behavior | History | Typical use |
|---|---|---|---|
| 0 | never changes | frozen | immutable codes, original signup date |
| 1 | overwrite in place | none | corrections, phone numbers |
| 2 | new row + validity range | full | segment, address, plan, ownership |
| 3 | previous value in an extra column | one step | simple before/after comparisons |
| 4 | separate history table | full, outside main dim | huge or audit-heavy dimensions |
| 6 | Type 2 + current value on every row | full + current | report by current and historical value |

Type 2 columns: new surrogate key per version, natural key, `valid_from`, `valid_to`, `is_current`, often `version`. Invariants:

+ Ranges are **half-open** `[valid_from, valid_to)`, contiguous, and non-overlapping.
+ Exactly one row per natural key has `is_current = true`.
+ The open row ends at a sentinel such as `9999-12-31`, not `NULL`, so range predicates work without `coalesce`.

Type 1 is cheap but destroys as-of reporting irreversibly; Type 3 answers "what was it before?" but not "what was it in March?". Type 6's current-value column must be rewritten on every version in the same transaction (write amplification). Applying Type 2 to everything makes every typo a version and bloats history nobody queries. When the business cannot state a requirement, default to Type 2 for analytical attributes and Type 1 for corrections.

```sql
-- Type 1: overwrite
update dim_customer as tgt set email = src.email
from stg_customer src where src.customer_id = tgt.customer_id and tgt.is_current;

-- Type 3: keep one previous value
update dim_customer as tgt
set previous_segment = tgt.segment, segment = src.segment, segment_changed_at = current_timestamp
from stg_customer src
where src.customer_id = tgt.customer_id and tgt.is_current
  and tgt.segment is distinct from src.segment;
```

## How do you implement SCD Type 2?

The load is two operations that must be atomic (one transaction or one `MERGE`), or readers can see zero or two current rows:

1. Stage the incoming snapshot and deduplicate to one row per natural key (latest wins).
2. Compare tracked attributes to the current row with a **null-safe** operator (`is distinct from`, or `<=>` in MySQL/Spark) so a `NULL -> value` change is detected.
3. Close changed current rows: `valid_to = run_ts`, `is_current = false`.
4. Insert a new version for every changed or brand-new key with `valid_from = run_ts`, `valid_to = '9999-12-31'`.

Idempotency: take `run_ts` from the batch (the snapshot's effective time), not from `now()` per row, and make change detection compare against the current row, so rerunning the same snapshot inserts nothing. The cell below implements this in DuckDB and proves the rerun is a no-op. On Snowflake, Databricks, or BigQuery the same logic is usually a single `MERGE` that unions changed rows in twice (once to close, once to insert).

In [ ]:
con.execute("""
create table dim_customer (
  customer_sk bigint primary key, customer_id varchar not null, segment varchar, email varchar,
  valid_from timestamp not null, valid_to timestamp not null, is_current boolean not null
);
insert into dim_customer values
  (1001, 'C1', 'standard', 'c1@x.com', '2026-01-01', '9999-12-31', true),
  (1002, 'C2', 'standard', null,       '2026-01-01', '9999-12-31', true);
create sequence customer_sk_seq start 1003;
create table stg_customer (customer_id varchar, segment varchar, email varchar, updated_at timestamp);
""")

TRACKED_CHANGED = "(d.segment is distinct from s.segment or d.email is distinct from s.email)"


def load_scd2(snapshot, run_ts):
    con.execute("delete from stg_customer")
    con.executemany("insert into stg_customer values (?, ?, ?, ?)", snapshot)
    con.execute("begin")
    con.execute("""
        create or replace temp table src as
        select * from stg_customer
        qualify row_number() over (partition by customer_id order by updated_at desc) = 1
    """)
    con.execute(f"""
        update dim_customer d set valid_to = ?, is_current = false
        from src s
        where d.customer_id = s.customer_id and d.is_current and {TRACKED_CHANGED}
    """, [run_ts])
    con.execute("""
        insert into dim_customer
        select nextval('customer_sk_seq'), s.customer_id, s.segment, s.email, ?, '9999-12-31', true
        from src s
        where not exists (select 1 from dim_customer d where d.customer_id = s.customer_id and d.is_current)
    """, [run_ts])
    con.execute("commit")


snapshot = [
    ("C1", "enterprise", "c1@x.com", "2026-10-01 08:00"),   # tracked change -> new version
    ("C2", "standard", "c2@x.com", "2026-10-01 08:00"),     # NULL -> value is a change too
    ("C3", "standard", "c3@x.com", "2026-10-01 08:00"),     # brand-new customer
]
load_scd2(snapshot, "2026-10-01")
rows_after_first = con.execute("select count(*) from dim_customer").fetchone()[0]
load_scd2(snapshot, "2026-10-02")                             # rerun: must be a no-op
rows_after_rerun = con.execute("select count(*) from dim_customer").fetchone()[0]

for row in con.execute("select * from dim_customer order by customer_id, valid_from").fetchall():
    print(row)
assert rows_after_first == rows_after_rerun == 5
assert con.execute("""
    select count(*) from (select customer_id from dim_customer
                          group by customer_id having count(*) filter (where is_current) <> 1)
""").fetchone()[0] == 0

## How do point-in-time (as-of) joins work with SCD2?

A fact row resolves to the version whose range contains the event time: `event_ts >= valid_from and event_ts < valid_to`. The half-open range guarantees adjacent versions never both match. Using `between valid_from and valid_to` makes the boundary instant match two versions and duplicates the fact.

Two ways to apply it:

+ **Resolve at load time**: look up the surrogate key when the fact is loaded, so reports join on a single equi-key. Best when facts arrive roughly in order.
+ **Resolve at query time**: range join on natural key plus validity window. Put the equality on the natural key first so it is not a cross product, and prune history by partition.

Don't use `is_current = true` as a shortcut for "as of" in a report that is rerun historically: next month's rerun returns a different answer for the same period, which fails audit.

In [ ]:
as_of = con.execute("""
    select f.order_id, f.line_number, f.order_ts, c.customer_sk, c.segment as segment_at_order
    from fact_order_line f
    join dim_customer c
      on c.customer_id = f.customer_id
     and f.order_ts >= c.valid_from
     and f.order_ts <  c.valid_to
    order by f.order_id, f.line_number
""").fetchall()
for row in as_of:
    print(row)

# exactly one version per fact row: the join preserves the fact row count
assert len(as_of) == 5
segments = {(r[0], r[1]): r[4] for r in as_of}
assert segments[("O1", 1)] == "standard"      # September order -> old version
assert segments[("O3", 1)] == "enterprise"    # October order -> new version

# inclusive BETWEEN double-matches an event exactly on the version boundary
boundary = con.execute("""
    select count(*) from dim_customer
    where customer_id = 'C1' and timestamp '2026-10-01' between valid_from and valid_to
""").fetchone()[0]
assert boundary == 2

## How do you handle late-arriving dimensions and facts?

Late data is normal: a fact for 2026-09-15 lands on 2026-10-04, or a customer appears in sales before the customer feed has them.

+ **Late fact, dimension known**: write the fact to the partition for its *event* date and resolve the dimension as of the event time. Never use load time as event time; it shifts metrics when pipelines are delayed and breaks partition pruning.
+ **Late fact, dimension unknown**: insert an **inferred member** (natural key, placeholder attributes, `is_inferred = true`) so the fact gets a valid key now; the next SCD load fills in the attributes. Track how many inferred rows remain.
+ **Late dimension change effective before already-loaded facts**: an out-of-order version. You must split the covering version's range and possibly re-point fact rows, which is a restatement. Store the source effective date and handle it through a controlled backfill rather than silently rewriting reported history.

Control the blast radius with a restatement window: accept late data for the last N days and rebuild those partitions; older corrections go through a documented backfill. Don't create a separate "late data" table: it doubles every downstream query and hides the gap.

```sql
-- inferred member: facts can join immediately, attributes filled in later
insert into dim_customer (customer_sk, customer_id, segment, valid_from, valid_to, is_current, is_inferred)
select nextval('customer_sk_seq'), s.customer_id, 'UNKNOWN', timestamp '1900-01-01', timestamp '9999-12-31', true, true
from (select distinct customer_id from stg_order) s
where not exists (select 1 from dim_customer d where d.customer_id = s.customer_id);
```

## How do you prevent double counting in analytical models?

Fan-out happens whenever a join is not many-to-one. The usual causes:

+ A dimension with duplicate keys (one row per address, or SCD2 history joined without a current or as-of filter). Referential integrity does not imply uniqueness.
+ A many-to-many relationship joined directly instead of through a bridge.
+ Two facts at different grains joined on a shared key (the **chasm trap**). Aggregate each to a common grain first.

Other guards: semi-additive measures use last value or average over time, never `sum`; `count(distinct key)` can hide a fan-out that `sum` exposes, so test measures, not just counts. Detection is mechanical: compare row count and measure total before and after each join.

In [ ]:
con.execute("""
create table dim_customer_address (customer_id varchar, address_id int, is_primary boolean);
insert into dim_customer_address values ('C1', 1, true), ('C1', 2, false), ('C2', 3, true);
""")

wrong = con.execute("""
    select sum(f.net_amount) from fact_order_line f
    join dim_customer_address a on a.customer_id = f.customer_id
""").fetchone()[0]
right = con.execute("""
    select sum(f.net_amount) from fact_order_line f
    join dim_customer_address a on a.customer_id = f.customer_id and a.is_primary
""").fetchone()[0]
print("fanned-out total:", wrong, "| correct total:", right)

# C1's lines (600) are counted twice because C1 has two address rows
assert right == 800
assert wrong == 800 + 600

## What is a bridge table and how do you model many-to-many relationships?

A bridge resolves a many-to-many relationship (orders with several promotions, accounts with several owners). Its grain is one row per relationship, keyed by the pair of surrogate keys. Two different metrics come out of it, and mixing them is the usual error:

+ **Weighted attribution**: `sum(measure * allocation_weight)`. Reconciles to the parent total only if each parent's weights sum to exactly 1, so test that.
+ **Influence**: `count(distinct parent_key)`, answering "how many orders were touched". It does not reconcile to revenue.

An unweighted join of the bridge to the fact counts revenue once per relationship. A "primary" flag lets simple reports pick one relationship. Name metrics by their formula so two dashboards don't both show "promotion revenue" computed differently.

In [ ]:
con.execute("""
create table fact_order as
select order_id, sum(net_amount) as revenue from fact_order_line group by order_id;
create table bridge_order_promotion (order_id varchar, promotion varchar, allocation_weight decimal(5, 4));
insert into bridge_order_promotion values
  ('O1', 'BACK2SCHOOL', 0.5), ('O1', 'LOYALTY', 0.5), ('O3', 'LOYALTY', 1.0);
""")

weighted = dict(con.execute("""
    select b.promotion, sum(f.revenue * b.allocation_weight)
    from fact_order f join bridge_order_promotion b using (order_id) group by all
""").fetchall())
unweighted_total = con.execute("""
    select sum(f.revenue) from fact_order f join bridge_order_promotion b using (order_id)
""").fetchone()[0]
influenced = con.execute("select count(distinct order_id) from bridge_order_promotion").fetchone()[0]
bad_weights = con.execute("""
    select order_id from bridge_order_promotion group by order_id having sum(allocation_weight) <> 1
""").fetchall()
print("weighted:", weighted, "| unweighted sum:", unweighted_total, "| influenced orders:", influenced)

promoted_revenue = con.execute("""
    select sum(revenue) from fact_order where order_id in (select order_id from bridge_order_promotion)
""").fetchone()[0]
assert sum(weighted.values()) == promoted_revenue == 600   # reconciles
assert unweighted_total == 780                               # O1 counted twice
assert influenced == 2 and bad_weights == []

## What is a factless fact table?

A factless fact records an event or relationship with no numeric measure. Aggregate it with `count(distinct ...)`, never by summing a placeholder `1` after a join (that double counts).

+ **Event factless**: a login, a class attendance.
+ **Coverage factless**: a relationship that was *possible*, such as a product on promotion or a customer eligible for a campaign. Anti-join it against the activity fact to answer "what did not happen".

Keep the same dimension keys and the same grain as the activity fact; comparing a monthly coverage fact against daily sales makes every product look unsold on most days.

In [ ]:
con.execute("""
create table fact_promotion_coverage (product_sk bigint, date_sk int, primary key (product_sk, date_sk));
insert into fact_promotion_coverage values (1, 20261015), (2, 20261015), (3, 20261015);
""")

unsold = con.execute("""
    select c.product_sk
    from fact_promotion_coverage c
    left join fact_order_line f on f.product_sk = c.product_sk and f.order_date_sk = c.date_sk
    where f.product_sk is null
""").fetchall()
print("promoted but unsold on 2026-10-15:", unsold)
assert unsold == [(3,)]

## What is a date dimension and a role-playing dimension?

A **date dimension** has one row per calendar day with precomputed attributes the business argues about: fiscal period, week start, `is_weekend`, `is_holiday`, the same day last year. Use an integer `date_sk` in `YYYYMMDD` form and reserve members for unknown and not-applicable dates. Without it, timezone and fiscal rules end up re-derived in every dashboard.

A **role-playing dimension** is one physical dimension joined several times with different meanings: `order_date_sk`, `ship_date_sk`, `delivery_date_sk` all point at `dim_date`; origin and destination both point at `dim_airport`. Join it with aliases (or thin views per role); never build physical copies that drift when a holiday is added to one.

```sql
select dd.year_month as order_month, ds.year_month as ship_month, count(*) as orders
from fact_order f
join dim_date dd on dd.date_sk = f.order_date_sk
join dim_date ds on ds.date_sk = f.ship_date_sk
group by 1, 2;
```

## How do you validate a data model after changes?

Validation is a set of invariants, each a query that must return zero rows, run as a pipeline step that fails the load:

+ Grain: the declared key is unique.
+ Dimension: one current row per natural key; no overlapping SCD2 ranges.
+ Referential integrity: no fact row points at a missing dimension key.
+ Reconciliation: mart totals equal source totals within tolerance (row counts alone miss wrong joins that change values but not cardinality).
+ Coverage: no unexplained date or entity gaps.

Test known edges too: a customer with no segment, a fact with a null key, a duplicate feed file. A test that only warns is a test that will be ignored.

In [ ]:
TESTS = {
    "grain_unique": """select order_id, line_number from fact_order_line
                       group by all having count(*) > 1""",
    "one_current_version": """select customer_id from dim_customer
                              group by customer_id having count(*) filter (where is_current) <> 1""",
    "no_overlapping_versions": """select a.customer_id from dim_customer a join dim_customer b
                                  on a.customer_id = b.customer_id and a.customer_sk < b.customer_sk
                                  and a.valid_from < b.valid_to and b.valid_from < a.valid_to""",
    "no_orphan_products": """select f.product_sk from fact_order_line f
                             left join dim_product p using (product_sk) where p.product_sk is null""",
}
results = {name: len(con.execute(sql).fetchall()) for name, sql in TESTS.items()}
print(results)
assert all(count == 0 for count in results.values())

## What is a metric definition layer and why does it matter?

Most metric disputes are modeling problems, not SQL problems. A metric contract (dbt semantic layer, Cube, LookML, or a governed table) pins down the expression, grain, filters, time dimension, allowed dimensions, owner, and additivity, so three dashboards cannot ship three revenue numbers.

```yaml
metric: net_revenue
owner: finance-analytics
base_table: gold.fact_order_line
grain: order_line
expression: sum(gross_amount - discount_amount - refund_amount)
filters: ["order_status <> 'cancelled'"]
time_dimension: order_date
allowed_dimensions: [dim_customer.segment, dim_customer.country_code, dim_product.category]
currency: USD
```

Additivity rules for contracts: ratios store numerator and denominator (`average_order_value = sum(net_revenue) / count(distinct order_id)`), distinct counts name the exact key, semi-additive measures name their time rollup. Averaging averages weights each row equally regardless of how many events it summarizes, so it is wrong whenever groups differ in size.

## What is a Data Vault model?

Data Vault 2.0 is an insert-only integration pattern built from three constructs:

+ **Hubs**: one row per business key ever seen, keyed by a hash of the key.
+ **Links**: relationships between hubs (order to customer).
+ **Satellites**: descriptive attributes with history (`load_ts`, `record_source`, `hash_diff` over tracked attributes) attached to a hub or link.

Every load is keyed by hash, so reruns insert nothing new, sources load in parallel, and `hash_diff` detects change without comparing every column. The cost: one entity is scattered across many tables, so BI directly on raw vault is painful. Teams publish point-in-time (PIT) and bridge tables, then dimensional marts on top. Data Vault is an integration layer, not a reporting model. Gotcha: the same attribute in two satellites (or in the hub and a satellite) makes change detection ambiguous.

In [ ]:
con.execute("""
create table hub_customer (customer_hk varchar primary key, customer_id varchar, load_ts timestamp, record_source varchar);
create table stg_crm_customer (customer_id varchar, load_ts timestamp);
insert into stg_crm_customer values ('C1', '2026-10-01'), ('C2', '2026-10-01'), ('C1', '2026-10-01');
""")


def load_hub():
    con.execute("""
        insert into hub_customer
        select md5(customer_id), customer_id, min(load_ts), 'crm'
        from stg_crm_customer s
        where not exists (select 1 from hub_customer h where h.customer_hk = md5(s.customer_id))
        group by customer_id
    """)
    return con.execute("select count(*) from hub_customer").fetchone()[0]


first_load, rerun = load_hub(), load_hub()
print("hub rows after load:", first_load, "| after rerun:", rerun)
assert first_load == rerun == 2   # duplicates collapsed, rerun idempotent

## Kimball vs Inmon: which modeling approach should you use?

| Aspect | Kimball (bottom-up) | Inmon (top-down) |
|---|---|---|
| Starting point | business processes and marts | enterprise-wide normalized model |
| Core structure | facts and dimensions | 3NF integration layer |
| Time to first value | fast (one mart) | slow (enterprise model first) |
| Consistency | conformed dimensions + bus matrix | central model |
| Best fit | BI-led, iterative delivery | regulated enterprises with many sources |

The pragmatic hybrid is the usual interview answer: land raw data (bronze), normalize and conform in silver (Inmon's discipline, or Data Vault for many-source parallel ingestion and audit), publish dimensional marts in gold (Kimball's delivery). Frame the choice by constraints, not ideology: audit and many sources favor a normalized core; time to value and self-service favor dimensional marts. Building marts straight from sources before entities are conformed leads each mart to invent its own customer.

## How do you model for performance and scale?

Performance is mostly decided by four modeling choices: grain, denormalization, physical layout, and SCD history volume.

### Pre-aggregation

Pre-aggregate when the same heavy group-by runs constantly and its result is small (500M order lines by category and month is ~48K rows). Keep the base fact at event grain; summaries are derived, named by grain (`fact_sales_monthly_category`), and store additive components only. A summary holding `avg_order_value` cannot be rolled up to a year correctly; store `sum_revenue` and `order_count` and divide at query time.

### Physical layout

+ **Partition** by event date or tenant so filters prune files. Never by a high-cardinality key like `order_id` (millions of tiny partitions).
+ **Cluster / sort** within partitions by common filter and join keys so zone maps skip blocks.
+ Keep files in a healthy size range and compact small files.
+ Align the model's join key with the physical sort key: joining on `customer_id` strings when the table is clustered on `customer_sk` cannot use the sort order.

```sql
-- Snowflake-style clustering on the columns the model filters and joins on
create or replace table fact_order_line (...) cluster by (order_date_sk, customer_sk);
-- Iceberg/Delta: partition by order_date, then sort or z-order by customer_sk
```

### SCD2 at scale

History grows with change rate, not entity count: 12M customers changing 2% per day add 240K rows/day, about 88M/year. Keep it manageable by partitioning history by `valid_from`, exposing a small current table or view, merging on the natural key before the range predicate, and archiving closed versions beyond retention. Full rebuilds every run eventually stop fitting in the batch window.

## Real world use cases for data modeling

| Domain | Pattern | Why |
|---|---|---|
| Retail, ~40M order lines/month | order-line star + weighted promotion bridge | margin by category/store/promotion without double counting; conformed date and product |
| Fintech, 7-year audit | SCD2 customer dim with as-of joins on transaction time | regulators need the risk tier that was true at the time; Type 1 would rewrite history |
| B2B SaaS, 2,000 tenants | `tenant_sk` on every fact and dimension + row-level security | partition pruning and isolation; cross-tenant joins can never match |
| Marketing | factless eligibility fact anti-joined to conversions | "eligible but did not convert" is about a relationship, not a measure |
| Logistics | accumulating snapshot per shipment | stage durations become `avg(days_in_transit)` instead of event-stream scans |

```sql
-- fintech: attribute values as of each transaction
select t.transaction_id, c.risk_tier as risk_tier_at_transaction, t.amount
from fact_transaction t
join dim_customer c
  on c.customer_id = t.customer_id
 and t.transaction_ts >= c.valid_from
 and t.transaction_ts <  c.valid_to;
```

## Data modeling exercises

Small standard-library calculations behind common modeling decisions. Each states the task and the expected result, then asserts it.

### Try it 1. Pick a grain from a requirement

**Task.** Find the single finest grain that answers all requirements. A requirement is answerable from its required level or finer (lower number). **Expected:** `order_line`.

In [ ]:
GRANULARITY = {"order_month": 3, "order": 2, "order_line": 1}
REQUIREMENTS = {
    "revenue by product by month": 2,
    "average order value": 2,
    "units per customer per day": 1,
}

finest_required = min(REQUIREMENTS.values())
valid = sorted(g for g, level in GRANULARITY.items() if level <= finest_required)
chosen = min(valid, key=lambda g: GRANULARITY[g])

print("finest level required:", finest_required, "| chosen grain:", chosen)
assert chosen == "order_line"

### Try it 2. Detect fan-out from join cardinalities

**Task.** Joined rows are `fact_rows_per_key x dim_rows_per_key` summed over keys; the measure inflates by the same ratio. **Expected:** 1,800 rows, 1.8x, 90,000 from a true 50,000.

In [ ]:
fact = {"total_rows": 1000, "total_amount": 50000}
fact_rows_per_key = {1: 400, 2: 400, 3: 200}
dim_rows_per_key = {1: 2, 2: 1, 3: 3}

joined_rows = sum(fact_rows_per_key[k] * dim_rows_per_key[k] for k in fact_rows_per_key)
ratio = joined_rows / fact["total_rows"]
inflated = fact["total_amount"] * ratio

print(f"joined rows = {joined_rows} ({ratio:.2f}x), inflated measure = {inflated:.0f}")
assert joined_rows == 1800 and ratio == 1.8 and inflated == 90000

### Try it 3. Star or snowflake from query patterns

**Task.** Compute weighted average join hops per query for each shape. **Expected:** star 1.0 vs snowflake 2.4, so star.

In [ ]:
QUERY_PATTERNS = {"single_dimension": 700, "two_hops": 200, "three_hops": 100}
STAR_HOPS = {"single_dimension": 1, "two_hops": 1, "three_hops": 1}
SNOWFLAKE_HOPS = {"single_dimension": 2, "two_hops": 3, "three_hops": 4}
total_queries = sum(QUERY_PATTERNS.values())


def weighted_hops(hops):
    return sum(QUERY_PATTERNS[p] * hops[p] for p in QUERY_PATTERNS) / total_queries


star_hops, snowflake_hops = weighted_hops(STAR_HOPS), weighted_hops(SNOWFLAKE_HOPS)
print(f"star = {star_hops:.2f}, snowflake = {snowflake_hops:.2f}")
assert round(star_hops, 2) == 1.0 and round(snowflake_hops, 2) == 2.4

### Try it 4. SCD2 history growth

**Task.** 12M customers, 2% change per day, 7-year retention. **Expected:** 240,000 rows/day, 87.6M/year, 613.2M retained.

In [ ]:
entities, change_rate, retention_years = 12_000_000, 0.02, 7

rows_per_day = entities * change_rate
rows_per_year = rows_per_day * 365
rows_retained = rows_per_year * retention_years

print(f"{rows_per_day:,.0f}/day, {rows_per_year:,.0f}/year, {rows_retained:,.0f} retained")
assert rows_per_day == 240_000 and rows_per_year == 87_600_000 and rows_retained == 613_200_000

### Try it 5. Size a bridge table

**Task.** 8M orders/month; 15% carry two promotions, 5% carry three. **Expected:** 0.45 relationships per order, 3.6M bridge rows/month, 43.2M/year.

In [ ]:
orders_per_month = 8_000_000
PROMOTIONS_PER_ORDER = {2: 0.15, 3: 0.05}

relationships_per_order = sum(count * share for count, share in PROMOTIONS_PER_ORDER.items())
bridge_rows_month = orders_per_month * relationships_per_order

print(f"{relationships_per_order:.2f} per order, {bridge_rows_month:,.0f}/month")
assert round(relationships_per_order, 2) == 0.45
assert round(bridge_rows_month) == 3_600_000 and round(bridge_rows_month * 12) == 43_200_000

### Try it 6. Detect a mixed grain

**Task.** Order-level shipping is stored on every order line. Compare the true total with the line-grain sum. **Expected:** 35 vs 115 (3.29x over-count).

In [ ]:
order_lines = {"O1": 3, "O2": 1, "O3": 4}
shipping_per_order = {"O1": 10, "O2": 5, "O3": 20}

true_total = sum(shipping_per_order.values())
naive_total = sum(shipping_per_order[o] * n for o, n in order_lines.items())

print(f"true = {true_total}, summed at line grain = {naive_total} ({naive_total / true_total:.2f}x)")
assert true_total == 35 and naive_total == 115